---

MRI analysis 

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# load relevant pgl classes
from pgl import pgl, pglDatabase, pglActions, pglFlywheel, pglAFNI, pglActionsMRI, pglActionsMNE, pglActionsFixes


---

Load behavioral data

---

In [ ]:
pgl.importSettings()

In [ ]:
newSession = pglActions.loadSession.execute()
if newSession: 
    session = newSession
    session.print()

---

Fix missing volume triggers

---

In [ ]:
fixVolumeTimes = pglActionsFixes.fixVolumeTimes()
newSession = fixVolumeTimes.execute(session=session)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.display()

In [ ]:
session.print()

---

Load data from flywheel

---

In [ ]:
newSession = pglActions.loadFlywheel().execute(session=session, showScanCounts=False)
if newSession: 
    session = newSession
    session.print()

In [ ]:
session.mri.print(match='nsdmini')

---

Align MRI scans with behavior/experiment session data

---

In [ ]:
session.setScanRunMatches([(4, 0), (5, 1), (6, 2), (10, 3), (11, 4), (12, 5), (17,6), (21, 7), (22, 8), (23, 9)])
#session.setScanRunMatches([(4, 0)])

In [ ]:
session.displayScanRunMapping()

---

Save the raw session to database

---

In [ ]:
session.saveRawSession()

---

Help on commands

---

In [ ]:
# Get a list of all actions within an action module
#pglActionsFixes.help()
#pglActions.help()
#pglActionsMNE.help()
pglActionsMRI.help()

In [ ]:
# Get specific help on a paritcular action
help(pglActionsMRI.mriEstimateMotionAFNI)

---

Database

---

In [ ]:
# display database
pglDatabase.print(experimentName='nsd-mini')
#pglDatabase.print()

---

Load a session

---

In [2]:
# load a specific session
#session = pglDatabase.loadSession(126)
session = pglDatabase.loadSession(128)

(pglDatabase:_reconstructSessionBundle) Restored 10 scan/run matches by acquisition identity; no matching action was replayed.
=================================== Session ====================================
  Source checkpoint : 128
  Runs              : 10
  Filesystem        : pglcheckpoint
  MEG/EEG data      : None
  MRI data          : 48 scans

---------------------------- Experiment and subject ----------------------------
  nsd-mini / sdbp_nsdmini_0002

------------------------------------ Run 0 -------------------------------------
Experiment: nsd-mini | Subject ID: sdbp_nsdmini_0002
Session: session_2026-11-13 | Run: run_12-28-39
2026-11-13T12:32:11.679832-08:00
Duration: 6 mins 3s
Number of volume triggers: 241
Median time between triggers: 1.510s
Mean ± std time between triggers: 1.510 ± 0.000000s
No interval timing discrepancies ≥100 ms detected, excluding any suspected manual final trigger.
taskName: NSD-mini (nTrials=120)

------------------------------------ Run 1 ----

---

Display information about a session


---

In [ ]:
# display the top-level information
session.print()

In [ ]:
# more detailed trial-by-trial information, for a particular run
session.print(runNum=0, verbose=True)

In [ ]:
# display mri scans
session.mri.print()

In [ ]:
# display behavioral plots
session.display();

---

Show history of actions

---

In [ ]:
session.history()

In [ ]:
# show more detailed information
session.history(verbose=True)

In [ ]:
# show what products actions produced
session.history(showProducts=True)

In [ ]:
# show the logs for a particular action
session.history(3,showLogs=True)

In [ ]:
# get the products for a particular action and print
motion = session.getProduct("motion")
motion.print()

In [ ]:
# dispaly any plots related to the product
motion.display();

---

AFNI: check availability and optional install

---

In [ ]:
# Install AFNI. If already installed, will not try to re-install but will display information about installation
afni = pglAFNI()
afni.install()

In [ ]:
# check availability of specifc commands
afni = pglAFNI(directory="~/proj/afni/abin")
available = afni.checkAvailable("3dvolreg")

---

Run motion comp

---

In [ ]:
newSession = pglActionsMRI.mriEstimateMotionAFNI().execute(session=session, deferApply=True)
if newSession: session = newSession

In [ ]:
# display matrices
motion = session.getProduct('motion')
if motion: motion.display()

In [ ]:
# display motion correction estimates
session.mri.

In [ ]:
session.saveSession()

In [ ]:
session.print(verbose=True)

In [ ]:
newSession = pglActionsFixes.reconstructVolumeTriggersFromNifti.execute(session=session)
if newSession: session = newSession

In [ ]:
newSession.print()

In [ ]:
session.runs[0].tasks[0].stimulusDatabase.print()

In [ ]:
session.runs[0].getTrialsByStimulus()

In [6]:
from pgl.pglSession import pglSession
from pgl.pglActionsMRI import pglActionsMRI
from pgl.pglPipeline import pglProductSelector

session = pglSession.loadSession(128)
if session is None:
    raise RuntimeError("Could not load checkpoint 128.")

motionApplication = pglActionsMRI.mriApplyTransforms()
motionApplication.configure(
    transforms=pglProductSelector(
        productID="c58099ba-696c-4095-b522-d27f63b1779d",
        kind="rigidMotion",
    ),
    outputName="motionCorrectedZarr",
    cacheDirectory="/Users/justin/.cache/pgl",
    keepIntermediates=False,
)

result = motionApplication.run(session)
if result is not session:
    raise RuntimeError("Motion correction failed; inspect the logs before retrying.")

print("Within-run motion correction completed successfully.")

(pglDatabase:_reconstructSessionBundle) Restored 10 scan/run matches by acquisition identity; no matching action was replayed.
=================================== Session ====================================
  Source checkpoint : 128
  Runs              : 10
  Filesystem        : pglcheckpoint
  MEG/EEG data      : None
  MRI data          : 48 scans

---------------------------- Experiment and subject ----------------------------
  nsd-mini / sdbp_nsdmini_0002

------------------------------------ Run 0 -------------------------------------
Experiment: nsd-mini | Subject ID: sdbp_nsdmini_0002
Session: session_2026-11-13 | Run: run_12-28-39
2026-11-13T12:32:11.679832-08:00
Duration: 6 mins 3s
Number of volume triggers: 241
Median time between triggers: 1.510s
Mean ± std time between triggers: 1.510 ± 0.000000s
No interval timing discrepancies ≥100 ms detected, excluding any suspected manual final trigger.
taskName: NSD-mini (nTrials=120)

------------------------------------ Run 1 ----

             0% [   00:00<       ?] Preparing local MRI file

Verifying SHA-256: /Users/justin/.cache/pgl/mri/08debde2ef575e5e0c8963e0a603d7aa9e922d20d1da2a598be7dbd0a2985f21-534969440/image.nii.gz
  Reading 510.2 MiB


             0% [   00:00<       ?] Hashing image.nii.gz

SHA-256 complete: image.nii.gz — 510.2 MiB in 0.3s
  Retrieving transform artifact: scan.45189148-dd13-4392-b049-81914845746d.matrices
  Retrieving transform artifact: scan.45189148-dd13-4392-b049-81914845746d.motionParameters
  Checking acquisition geometry and sampling transforms.
  Constructing lazy source metadata; no voxel computation is expected.
  Preflight complete: 1 of 10 scans.

Motion preflight: scan 5 'nsdmini_run02_bold_1p4mm_R3x3_esp570_TR1510: 1_3_12_2_1107_5_2_61_251202_2026111312390826805148970_0_0_0.nii.gz' (1 of 10 completed).
  Resolving original acquisition and checking its identity/header.


             0% [   00:00<       ?] Preparing local MRI file

... (74 outputs left out)

SHA-256 complete: image.nii.gz — 508.1 MiB in 0.3s
  Verifying 20 transform artifacts.

Starting correction and Zarr publication. The progress bar measures Dask tasks, not bytes or volumes.


Traceback (most recent call last):
  File "/Users/justin/proj/pgl/pgl/pglPipeline.py", line 2137, in run
    result = self._run(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/justin/proj/pgl/pgl/pglActionsMRI.py", line 1754, in _run
    pglActionsMRI._applyWithinRunMotion(
  File "/Users/justin/proj/pgl/pgl/pglActionsMRI.py", line 1294, in _applyWithinRunMotion
    registered = session.mri.publishImageReplacements(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/justin/proj/pgl/pgl/pglMRI.py", line 2438, in publishImageReplacements
    pglMessages.message(f"Writing replacement image {candidate['scanUUID']} ({imagePosition} of {len(prepared)} completed): shape={image.shape}, dtype={image.dtype}.")
    ^^^^^^^^^^^
NameError: name 'pglMessages' is not defined


❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(mriApplyTransforms->pglAction:run) Error running action mriApplyTransforms: name 'pglMessages' is not defined
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌


RuntimeError: Motion correction failed; inspect the logs before retrying.

In [4]:
import dask.array as da
import numpy as np
import xarray as xr
from dask import delayed
from dask.callbacks import Callback

from pgl.pglData import pglDataMatrix


class rejectUnexpectedCompute(Callback):
    def _start(self, graph):
        raise AssertionError("Unexpected Dask computation during matrix attachment!")


@delayed
def makeValues():
    return np.arange(24, dtype=np.float32).reshape(2, 3, 4)


lazyValues = da.from_delayed(
    makeValues(),
    shape=(2, 3, 4),
    dtype=np.float32,
)
first = xr.DataArray(lazyValues, dims=("x", "y", "z"))
second = xr.DataArray(lazyValues + 1, dims=("x", "y", "z"))
dataset = xr.Dataset({"image": first})

notifications = []

with rejectUnexpectedCompute():
    # Construction must not compute.
    matrix = pglDataMatrix(first, name="lazy attachment test")
    assert matrix.data is first

    matrix.observe(
        lambda change: notifications.append(change),
        names="data",
    )

    # Same object: no notification or numerical comparison.
    matrix.data = first
    assert len(notifications) == 0

    # Different object: exactly one notification, without computation.
    matrix.data = second
    assert len(notifications) == 1
    assert notifications[0]["old"] is first
    assert notifications[0]["new"] is second

    # Dataset and None assignments must also remain noncomputing.
    matrix.data = dataset
    matrix.data = None
    matrix.data = first
    assert len(notifications) == 4

    # Existing type validation must still work and leave the old value intact.
    try:
        matrix.data = "not scientific xarray data"
    except TypeError:
        pass
    else:
        raise AssertionError("Invalid attachment was accepted.")

    assert matrix.data is first
    assert len(notifications) == 4

# Explicit computation is still allowed and produces the correct values.
np.testing.assert_array_equal(
    matrix.data.compute().data,
    np.arange(24, dtype=np.float32).reshape(2, 3, 4),
)

print("PASS: construction and reassignment never compute arrays; validation and notifications remain intact.")

PASS: construction and reassignment never compute arrays; validation and notifications remain intact.


In [ ]:
def runActionWithDiagnostics(action, session):
    """Run normally while recording periodic Python stack snapshots.

    Diagnostic only: no changes to the action's scientific processing.
    Uses Python's process-wide fault-handler timer; do not combine with
    another active dump_traceback_later() diagnostic.
    """
    import faulthandler
    from pathlib import Path
    from uuid import uuid4

    directory = Path.home() / ".cache" / "pgl" / "actionDiagnostics"
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f"stacks-{uuid4().hex[:12]}.log"

    print(f"Stack diagnostics: {path}", flush=True)
    print(f"In a separate Terminal, run:\n  tail -f '{path}'", flush=True)

    with path.open("x", encoding="utf-8") as handle:
        faulthandler.dump_traceback_later(20, repeat=True, file=handle)
        try:
            return action.run(session)
        finally:
            faulthandler.cancel_dump_traceback_later()

    # The diagnostic file is deliberately retained.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch
from uuid import uuid4

import dask.array as da
import nibabel as nib
import numpy as np
from dask.callbacks import Callback

from pgl.pglMRI import pglMRIData


class rejectPreflightCompute(Callback):
    def _start(self, graph):
        raise AssertionError("MRI construction unexpectedly started Dask computation.")


def rejectVoxelRead(*args, **kwargs):
    raise AssertionError("MRI construction unexpectedly read NIfTI voxel values.")


with TemporaryDirectory(prefix="pgl-compressed-laziness-") as temporary:
    filename = Path(temporary) / "acquisition.nii.gz"
    expected = np.arange(120, dtype=np.float32).reshape(2, 3, 4, 5)

    image = nib.Nifti1Image(expected, np.eye(4))
    image.header.set_xyzt_units("mm", "sec")
    image.header.set_zooms((1.0, 1.0, 1.0, 1.5))
    nib.save(image, filename)

    scanUUID = str(uuid4())

    # Both protections apply only during construction/structural inspection.
    with rejectPreflightCompute():
        with patch.object(nib.arrayproxy.ArrayProxy, "__getitem__", rejectVoxelRead):
            with patch.object(nib.arrayproxy.ArrayProxy, "__array__", rejectVoxelRead):
                matrix = pglMRIData.fromNifti(
                    filename,
                    scanUUID=scanUUID,
                    sourceStageID=f"acquisition:{scanUUID}",
                )
                pglMRIData.validate(matrix)
                view = pglMRIData.getDataMatrix(matrix)

                assert isinstance(matrix.data.data, da.Array)
                assert isinstance(view.data, da.Array)
                assert view.shape == (24, 5)

    try:
        # Explicitly requested computation must still read the correct values.
        np.testing.assert_array_equal(
            matrix.data.isel(volume=2).compute().data,
            expected[..., 2],
        )
        print("PASS: compressed NIfTI construction and matrix-view creation read no voxels and execute no Dask tasks.")
        print("PASS: explicitly computing one volume returns the correct values.")
    finally:
        matrix.close()

PASS: compressed NIfTI construction and matrix-view creation read no voxels and execute no Dask tasks.
PASS: explicitly computing one volume returns the correct values.


In [1]:
import gzip
import hashlib
from pathlib import Path
from tempfile import TemporaryDirectory

import nibabel as nib
import numpy as np

from pgl.pglMRI import pglMRIData


with TemporaryDirectory(prefix="pgl-volume-source-test-") as temporary:
    root = Path(temporary)
    compressedPath = root / "acquisition.nii.gz"

    rawValues = np.arange(120, dtype=np.int16).reshape(2, 3, 4, 5)
    image = nib.Nifti1Image(rawValues, np.eye(4))
    image.header.set_xyzt_units("mm", "sec")
    image.header.set_zooms((1.0, 1.0, 1.0, 1.5))
    image.header.set_slope_inter(2.0, 7.0)
    nib.save(image, compressedPath)

    originalIdentity = hashlib.sha256(compressedPath.read_bytes()).hexdigest()

    workingPath = pglMRIData.prepareVolumeSource(
        compressedPath,
        cacheDirectory=root / "cache",
    )

    assert workingPath.suffix == ".nii"
    assert workingPath != compressedPath

    # Tiny fixture only: compare the entire decompressed byte stream exactly.
    with gzip.open(compressedPath, "rb") as source:
        expectedBytes = source.read()
    assert workingPath.read_bytes() == expectedBytes

    original = nib.load(compressedPath)
    prepared = nib.load(workingPath, mmap=True)

    assert prepared.shape == original.shape
    assert prepared.get_data_dtype() == original.get_data_dtype()
    assert prepared.dataobj.slope == original.dataobj.slope
    assert prepared.dataobj.inter == original.dataobj.inter

    # Unscaled .nii payload can be memory-mapped instead of gzip-decoded.
    unscaled = prepared.dataobj.get_unscaled()
    assert isinstance(unscaled, np.memmap)
    np.testing.assert_array_equal(unscaled, rawValues)

    for volumeIndex in (0, 2, 4):
        values = np.asarray(prepared.dataobj[..., volumeIndex])
        np.testing.assert_array_equal(
            values,
            rawValues[..., volumeIndex] * 2.0 + 7.0,
        )
        np.testing.assert_array_equal(
            values,
            np.asarray(original.dataobj[..., volumeIndex]),
        )

    # An already uncompressed input is returned without another copy.
    assert pglMRIData.prepareVolumeSource(
        workingPath,
        cacheDirectory=root / "cache",
    ) == workingPath

    assert hashlib.sha256(compressedPath.read_bytes()).hexdigest() == originalIdentity

    # Release the test mapping before the temporary directory is removed.
    del unscaled, prepared, original

print("PASS: exact decompression, memory-mapped access, preserved scaling, and unchanged acquisition.")

Preparing memory-mapped volume source: /var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-volume-source-test-gf1qtx97/acquisition.nii.gz
  Uncompressed working copy: /var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-volume-source-test-gf1qtx97/cache/mriVolumeSources/source-2f8cxjm0/image.nii


             0% [   00:00<       ?] Decompressing MRI acquisition once

Volume source ready: 0.0 MiB in 0.1s
PASS: exact decompression, memory-mapped access, preserved scaling, and unchanged acquisition.


In [7]:
import time

from pgl.pglProgress import pglProgress

with pglProgress("Testing shared action progress") as progress:
    with progress.section(0.0, 0.3, label="Preflight") as preflight:
        preflight(0, "Checking inputs")
        time.sleep(0.3)
        preflight.finish("Inputs verified")

    # Finishing/closing a child must not finish/close the parent.
    assert not progress._finished
    assert not progress._closed
    assert abs(progress._fraction - 0.3) < 1e-9

    with progress.section(0.3, 0.9, label="Correction") as correction:
        for completed in range(1, 6):
            correction(completed / 5, f"Processing volumes — {completed}/5 completed")
            time.sleep(0.2)

        correction.finish("Correction written")

    assert not progress._finished
    assert not progress._closed
    assert abs(progress._fraction - 0.9) < 1e-9

    # A handled child failure must not destroy the parent's display.
    try:
        with progress.section(0.9, 0.95, label="Validation") as validation:
            validation(0, "Testing recoverable child exception")
            raise ValueError("Intentional child-scope test")
    except ValueError:
        pass

    assert not progress._closed
    assert not progress._finished

    progress(0.99, "Finalizing")
    time.sleep(0.3)
    progress.finish("Shared progress test complete")

assert progress._finished
assert progress._closed
print("PASS: one display; child completion and closure do not finish the parent.")

             0% [   00:00<       ?] Testing shared action progress

PASS: one display; child completion and closure do not finish the parent.


In [ ]:
import ast
import os
import tempfile
from datetime import datetime
from pathlib import Path

import pgl


def installSharedActionProgress():
    """Install reporting-only changes with backups and syntax validation."""
    packagePath = Path(pgl.__file__).resolve().parent
    helperPath = packagePath / "pglActionProgress.py"

    if helperPath.exists():
        raise RuntimeError(
            f"{helperPath} already exists. Stop here rather than applying this patch twice."
        )

    helperSource = r'''
################################################################
#   filename: pglActionProgress.py
#    purpose: Shared action progress without duplicate displays
################################################################

from contextlib import contextmanager, nullcontext
from contextvars import ContextVar
from functools import wraps


_activeProgress = ContextVar("pglActiveProgress", default=None)


def currentProgress():
    """Return the current explicitly assigned progress scope, or None."""
    return _activeProgress.get()


def _statusSection(progress):
    """Borrow a zero-width section for a helper without assigned work weight."""
    child = progress.section(0, 0)
    position = min(progress._end, max(progress._start, progress._root._fraction))
    child._start = position
    child._end = position
    return child


def borrowProgress(instance, description):
    """Initialize a nested pglProgress as a borrowed status-only section.

    Explicit weighted scopes advance percentages. Incidental helper bars only
    update status, so a completed file cannot finish the enclosing action.
    Returns False when there is no active action/operation display.
    """
    active = currentProgress()
    if active is None:
        return False

    child = _statusSection(active)
    instance._root = child._root
    instance._start = child._start
    instance._end = child._end
    instance._label = child._label
    instance(None, description)
    return True


def statusOutput(*values, sep=" ", end="\n", flush=False, **kwargs):
    """Route routine processing descriptions to progress, not the console.

    Used only by explicitly migrated processing helpers. It is not a global
    stdout filter: AFNI output, warnings, and tracebacks remain ordinary output.
    """
    text = " ".join(sep.join(str(value) for value in values).split())
    progress = currentProgress()

    if progress is not None:
        progress(None, text)


def noProgressBar(*args, **kwargs):
    """Compatibility context replacing removed Dask display contexts."""
    return nullcontext()


def progressCall(progress, start, end, description, function, *args, **kwargs):
    """Execute a helper in an explicit weighted section of one shared display."""
    child = progress.section(start, end)
    token = _activeProgress.set(child)

    try:
        child(0, description)
        result = function(*args, **kwargs)
        child(1, description)
        return result
    finally:
        _activeProgress.reset(token)


def operationProgress(description):
    """Give an independently called operation a bar, or borrow its caller's.

    Within an action, reserve the last 5% of the assigned scope for the owning
    caller's finalization. An operation never declares overall action success.
    """
    def decorate(function):
        @wraps(function)
        def wrapped(*args, **kwargs):
            from .pglProgress import pglProgress

            parent = currentProgress()
            owned = parent is None
            root = pglProgress(description) if owned else None
            scope = root.section(0, 0.95) if owned else parent.section(0, 0.95)
            token = _activeProgress.set(scope)
            outcome = "Failed"

            try:
                scope(0, description)
                result = function(*args, **kwargs)
                if owned:
                    root.finish(f"{description} complete")
                outcome = None
                return result
            except KeyboardInterrupt:
                outcome = "Interrupted"
                raise
            finally:
                _activeProgress.reset(token)
                if owned:
                    root.close(outcome)

        return wrapped
    return decorate


def actionProgress(function):
    """Own one display around the complete existing action runner.

    Includes history commitment, retention, presentation, and console-log
    attachment because those remain inside the wrapped run() implementation.

    Nested actions borrow the display without claiming their parent's work
    fraction. They do not close it or declare overall completion.

    Does not change action outcomes, recorder contexts, or scientific code.
    """
    @wraps(function)
    def wrapped(self, *args, **kwargs):
        from .pglPipeline import pglActionStatus
        from .pglProgress import pglProgress

        # Preserve existing refusals without opening a misleading new bar.
        if getattr(self, "_executionInProgress", False) or not self.isConfigured():
            return function(self, *args, **kwargs)

        parent = currentProgress()
        owned = parent is None
        root = pglProgress(self.name) if owned else None
        scope = root.section(0, 0.99) if owned else _statusSection(parent)
        token = _activeProgress.set(scope)
        previousCount = len(self.actionHistory)
        outcome = "Failed"

        try:
            scope(0, f"{self.name} — starting")
            result = function(self, *args, **kwargs)

            if len(self.actionHistory) == previousCount:
                outcome = "Not started"
            elif self.status == pglActionStatus.SUCCESS:
                if owned:
                    root.finish(f"{self.name} — complete")
                outcome = None
            elif self.status == pglActionStatus.SKIPPED:
                outcome = "Skipped"
            else:
                outcome = "Failed"

            return result

        except KeyboardInterrupt:
            outcome = "Interrupted"
            raise

        finally:
            _activeProgress.reset(token)
            if owned:
                root.close(outcome)

    return wrapped


@contextmanager
def daskProgress(progress, description):
    """Update the existing display from Dask task completions.

    Creates no Dask/tqdm display. Fractions describe tasks, not bytes or elapsed
    time. The motion publisher uses the synchronous scheduler, so these updates
    run on the computation's calling thread.

    Graph completion reserves the final fraction for writer return/cleanup.
    """
    from dask.callbacks import Callback

    class ReportTasks(Callback):
        def _start(self, graph):
            self.total = max(len(graph), 1)
            self.completed = 0
            progress(0, f"{description} — computing and writing")

        def _posttask(self, key, result, graph, state, workerID):
            self.completed += 1
            progress(
                min(0.98, self.completed / self.total * 0.98),
                f"{description} — {self.completed}/{self.total} tasks completed",
            )

    with ReportTasks():
        yield

    progress(1, f"{description} — write completed")
'''.lstrip()

    imports = (
        "from .pglActionProgress import "
        "actionProgress, operationProgress, currentProgress, "
        "progressCall, statusOutput, noProgressBar, daskProgress\n"
    )

    class SourceEditor:
        """Apply checked source-span edits while preserving untouched comments."""

        def __init__(self, path):
            self.path = path
            self.source = path.read_text(encoding="utf-8")
            self.lines = self.source.splitlines(keepends=True)
            self.offsets = [0]
            for line in self.lines:
                self.offsets.append(self.offsets[-1] + len(line))
            self.tree = ast.parse(self.source)
            self.edits = []

        def position(self, line, byteColumn):
            # AST columns are UTF-8 byte offsets; Python string offsets are not.
            prefix = self.lines[line - 1].encode("utf-8")[:byteColumn].decode("utf-8")
            return self.offsets[line - 1] + len(prefix)

        def span(self, node):
            return (
                self.position(node.lineno, node.col_offset),
                self.position(node.end_lineno, node.end_col_offset),
            )

        def text(self, node):
            start, end = self.span(node)
            return self.source[start:end]

        def replace(self, node, replacement):
            start, end = self.span(node)
            self.edits.append((start, end, replacement))

        def insertBeforeLine(self, line, text):
            offset = self.offsets[line - 1]
            self.edits.append((offset, offset, text))

        def insertAfterLine(self, line, text):
            offset = self.offsets[line]
            self.edits.append((offset, offset, text))

        def method(self, className, methodName):
            classes = [
                node for node in self.tree.body
                if isinstance(node, ast.ClassDef) and node.name == className
            ]
            if len(classes) != 1:
                raise RuntimeError(f"Expected one {className} in {self.path.name}.")

            methods = [
                node for node in classes[0].body
                if isinstance(node, ast.FunctionDef) and node.name == methodName
            ]
            if len(methods) != 1:
                raise RuntimeError(f"Expected one {className}.{methodName}.")
            return methods[0]

        def addImport(self, statement):
            # Keep module docstrings and future imports in their required places.
            index = 0
            if (
                self.tree.body
                and isinstance(self.tree.body[0], ast.Expr)
                and isinstance(self.tree.body[0].value, ast.Constant)
                and isinstance(self.tree.body[0].value.value, str)
            ):
                index = 1

            while (
                index < len(self.tree.body)
                and isinstance(self.tree.body[index], ast.ImportFrom)
                and self.tree.body[index].module == "__future__"
            ):
                index += 1

            if index < len(self.tree.body):
                self.insertBeforeLine(self.tree.body[index].lineno, statement)
            else:
                self.edits.append((len(self.source), len(self.source), "\n" + statement))

        def decorate(self, function, decorator):
            indentation = " " * function.col_offset
            # Inserting directly before def keeps staticmethod/classmethod
            # outside the new function decorator.
            self.insertBeforeLine(function.lineno, f"{indentation}@{decorator}\n")

        def addLocalProgress(self, function):
            first = function.body[0]
            indentation = " " * (function.col_offset + 4)

            if (
                isinstance(first, ast.Expr)
                and isinstance(first.value, ast.Constant)
                and isinstance(first.value.value, str)
            ):
                self.insertAfterLine(first.end_lineno, f"{indentation}_progress = currentProgress()\n")
            else:
                self.insertBeforeLine(first.lineno, f"{indentation}_progress = currentProgress()\n")

        def render(self):
            """Combine same-position insertions before applying source edits.

            Preserves insertion request order. In particular, a new with-line
            must precede the indentation added to the original statement.
            Validates all spans before compiling the generated source.
            """
            insertions = {}
            replacements = []

            for start, end, replacement in self.edits:
                if not 0 <= start <= end <= len(self.source):
                    raise RuntimeError(f"Invalid source-edit range in {self.path.name}.")

                if start == end:
                    insertions.setdefault(start, []).append(replacement)
                else:
                    replacements.append((start, end, replacement))

            # Nonempty replacements must not overlap.
            ordered = sorted(replacements, key=lambda edit: (edit[0], edit[1]))
            for left, right in zip(ordered, ordered[1:]):
                if right[0] < left[1]:
                    raise RuntimeError(f"Overlapping replacements in {self.path.name}.")

            # An insertion may occur at a replacement boundary, but not inside
            # text that another edit replaces.
            for position in insertions:
                if any(start < position < end for start, end, _ in replacements):
                    raise RuntimeError(f"Insertion overlaps a replacement in {self.path.name}.")

            # Combine insertions at the same position in the order requested.
            # Applying them independently in reverse source order would reverse
            # their contents and can indent the new with-line incorrectly.
            edits = list(replacements)
            edits.extend(
                (position, position, "".join(parts))
                for position, parts in insertions.items()
            )

            result = self.source

            # At the same starting position, apply a replacement first, then
            # prepend any combined insertion.
            for start, end, replacement in sorted(
                edits,
                key=lambda edit: (edit[0], edit[1]),
                reverse=True,
            ):
                result = result[:start] + replacement + result[end:]

            compile(result, str(self.path), "exec")
            return result

    editors = {
        name: SourceEditor(packagePath / name)
        for name in ("pglProgress.py", "pglPipeline.py", "pglMRI.py", "pglActionsMRI.py", "pglExternalTools.py")
    }

    def calls(function, name):
        return [
            node for node in ast.walk(function)
            if isinstance(node, ast.Call)
            and (
                isinstance(node.func, ast.Name) and node.func.id == name
                or isinstance(node.func, ast.Attribute) and node.func.attr == name
            )
        ]

    def uniqueCall(function, name):
        found = calls(function, name)
        if len(found) != 1:
            raise RuntimeError(f"Expected one {name} call in {function.name}; found {len(found)}.")
        return found[0]

    def wrapCall(editor, node, start, end, description):
        functionText = editor.text(node.func)
        arguments = [editor.text(arg) for arg in node.args]

        for keyword in node.keywords:
            value = editor.text(keyword.value)
            arguments.append(f"**{value}" if keyword.arg is None else f"{keyword.arg}={value}")

        suffix = ", " + ", ".join(arguments) if arguments else ""
        editor.replace(
            node,
            f"progressCall(_progress, {start}, {end}, {description}, {functionText}{suffix})",
        )

    def routeStatus(editor, function):
        """Only explicitly selected helper output is changed; no global filter."""
        count = 0
        for node in ast.walk(function):
            if not isinstance(node, ast.Call): continue

            isPrint = isinstance(node.func, ast.Name) and node.func.id == "print"
            isMessage = (
                isinstance(node.func, ast.Attribute)
                and isinstance(node.func.value, ast.Name)
                and node.func.value.id == "pglMessages"
                and node.func.attr in ("message", "print")
            )
            if isPrint or isMessage:
                editor.replace(node.func, "statusOutput")
                count += 1

            if isinstance(node.func, ast.Name) and node.func.id == "ProgressBar":
                editor.replace(node.func, "noProgressBar")

        return count

    # ---------------------------------------------------------------
    # Shared pglProgress construction: incidental helper bars borrow
    # the active display and cannot advance its overall percentage.
    # ---------------------------------------------------------------
    editor = editors["pglProgress.py"]
    initializer = editor.method("pglProgress", "__init__")
    finish = editor.text(editor.method("pglProgress", "finish"))
    close = editor.text(editor.method("pglProgress", "close"))

    if "self is not root" not in finish or "self is not root" not in close:
        raise RuntimeError("Install the tested child-safe finish()/close() updates first.")

    editor.addImport("from .pglActionProgress import borrowProgress\n")
    first = initializer.body[0]
    insertion = (
        "        # Nested helpers share the active action display.\n"
        "        if borrowProgress(self, description): return\n"
    )
    if isinstance(first, ast.Expr) and isinstance(first.value, ast.Constant) and isinstance(first.value.value, str):
        editor.insertAfterLine(first.end_lineno, insertion)
    else:
        editor.insertBeforeLine(first.lineno, insertion)

    # ---------------------------------------------------------------
    # Root action scope surrounds the existing runner, including its
    # final console-log attachment. No runner internals are replaced.
    # ---------------------------------------------------------------
    editor = editors["pglPipeline.py"]
    editor.addImport(imports)
    run = editor.method("pglAction", "run")
    editor.decorate(run, "actionProgress")

    # Retention success counts are routine status, not scientific warnings.
    retention = editor.method("pglAction", "_finalizeImageRetention")
    for node in calls(retention, "message"):
        if isinstance(node.func, ast.Attribute) and isinstance(node.func.value, ast.Name) and node.func.value.id == "pglMessages":
            editor.replace(node.func, "statusOutput")

    # ---------------------------------------------------------------
    # Hashing uses its existing byte checks and pglProgress. Its extra
    # console announcements are replaced by shared display updates.
    # ---------------------------------------------------------------
    editor = editors["pglExternalTools.py"]
    editor.addImport(imports)
    hashing = editor.method("pglExternalTool", "_fileIdentity")
    routeStatus(editor, hashing)

    # ---------------------------------------------------------------
    # Decompression and batch publication reuse the action display.
    # Publication receives per-image Dask task progress, with no new bar.
    # ---------------------------------------------------------------
    editor = editors["pglMRI.py"]
    editor.addImport(imports)

    preparation = editor.method("pglMRIData", "prepareVolumeSource")
    routeStatus(editor, preparation)

    publisher = editor.method("pglMRI", "publishImageReplacements")
    editor.decorate(publisher, 'operationProgress("Publishing MRI replacements")')
    editor.addLocalProgress(publisher)
    routeStatus(editor, publisher)

    saveCall = uniqueCall(publisher, "saveDataMatrix")
    saveStatements = [
        node for node in ast.walk(publisher)
        if isinstance(node, ast.Assign) and node.value is saveCall
    ]
    if len(saveStatements) != 1:
        raise RuntimeError("Could not identify the publisher's save assignment.")
    saveStatement = saveStatements[0]

    writerLoops = [
        node for node in ast.walk(publisher)
        if isinstance(node, ast.For)
        and any(child is saveStatement for child in ast.walk(node))
    ]
    if len(writerLoops) != 1:
        raise RuntimeError("Could not identify the single image-publication loop.")
    writerLoop = writerLoops[0]

    loopLine = editor.lines[writerLoop.lineno - 1]
    if loopLine.strip() == "for item in prepared:":
        offset = editor.offsets[writerLoop.lineno - 1]
        editor.edits.append((
            offset,
            offset + len(loopLine),
            " " * writerLoop.col_offset + "for imagePosition, item in enumerate(prepared):\n",
        ))
        imageIndex = "imagePosition"
    elif (
        isinstance(writerLoop.target, ast.Tuple)
        and len(writerLoop.target.elts) == 2
        and isinstance(writerLoop.target.elts[0], ast.Name)
        and isinstance(writerLoop.target.elts[1], ast.Name)
        and writerLoop.target.elts[1].id == "item"
        and isinstance(writerLoop.iter, ast.Call)
        and isinstance(writerLoop.iter.func, ast.Name)
        and writerLoop.iter.func.id == "enumerate"
        and len(writerLoop.iter.args) == 1
        and isinstance(writerLoop.iter.args[0], ast.Name)
        and writerLoop.iter.args[0].id == "prepared"
    ):
        imageIndex = writerLoop.target.elts[0].id
    else:
        raise RuntimeError("Unexpected image-publication loop; no files were changed.")

    indentation = " " * (writerLoop.col_offset + 4)
    editor.insertAfterLine(
        writerLoop.lineno,
        indentation
        + f"_imageProgress = _progress.section({imageIndex} / len(prepared), ({imageIndex} + 1) / len(prepared))\n",
    )

    # Add a task callback around the existing writer, preserving its arguments.
    if saveStatement.lineno != saveStatement.end_lineno:
        raise RuntimeError("Expected the existing single-line saveDataMatrix assignment.")

    indent = " " * saveStatement.col_offset
    editor.insertBeforeLine(
        saveStatement.lineno,
        indent
        + 'with daskProgress(_imageProgress.section(0, 0.95), f"Writing image {candidate[\'scanUUID\']}"):\n',
    )
    editor.edits.append((
        editor.offsets[saveStatement.lineno - 1],
        editor.offsets[saveStatement.lineno - 1],
        "    ",
    ))

    declarationCalls = [
        node for node in ast.walk(writerLoop)
        if isinstance(node, ast.Expr)
        and isinstance(node.value, ast.Call)
        and isinstance(node.value.func, ast.Attribute)
        and isinstance(node.value.func.value, ast.Name)
        and node.value.func.value.id == "declarations"
        and node.value.func.attr == "append"
    ]
    if len(declarationCalls) != 1:
        raise RuntimeError("Could not identify completed image-declaration registration.")

    declarationStatement = declarationCalls[0]
    editor.insertAfterLine(
        declarationStatement.end_lineno,
        " " * declarationStatement.col_offset
        + '_imageProgress(1, f"Image {candidate[\'scanUUID\']} — written and validated")\n',
    )

    # ---------------------------------------------------------------
    # Motion helper: explicit weighted stages on the shared display.
    # Numerical interpolation, integrity checks and publication stay intact.
    # ---------------------------------------------------------------
    editor = editors["pglActionsMRI.py"]
    editor.addImport(imports)
    application = editor.method("pglActionsMRI", "_applyWithinRunMotion")
    editor.decorate(application, 'operationProgress("Applying MRI motion correction")')
    editor.addLocalProgress(application)
    routeStatus(editor, application)

    requiredLocals = {"scanPosition", "selectedUUIDs", "position", "jobs", "source"}
    localNames = {
        node.id for node in ast.walk(application)
        if isinstance(node, ast.Name)
    }
    if not requiredLocals <= localNames:
        raise RuntimeError("Expected the latest memory-mapped motion helper. No files were changed.")

    wrapCall(
        editor,
        uniqueCall(application, "_resolveImageSource"),
        "0.04 + 0.16 * scanPosition / len(selectedUUIDs)",
        "0.04 + 0.16 * (scanPosition + 1) / len(selectedUUIDs)",
        'f"Preflight scan {scanIndex} — retrieving and verifying acquisition"',
    )

    wrapCall(
        editor,
        uniqueCall(application, "prepareVolumeSource"),
        "0.20 + 0.15 * position / len(jobs)",
        "0.20 + 0.15 * (position + 1) / len(jobs)",
        'f"Preparing scan {source[\'scanIndex\']} — uncompressed volume access"',
    )

    verificationCalls = sorted(
        calls(application, "requireSourceBytesUnchanged"),
        key=lambda node: node.lineno,
    )
    if len(verificationCalls) != 2:
        raise RuntimeError("Expected the before/after source-verification calls.")

    wrapCall(
        editor, verificationCalls[0], "0.35", "0.40",
        '"Verifying source bytes before correction"',
    )
    wrapCall(
        editor, verificationCalls[1], "0.92", "0.99",
        '"Final source and transform verification"',
    )
    wrapCall(
        editor,
        uniqueCall(application, "publishImageReplacements"),
        "0.40", "0.92",
        '"Correcting volumes and writing Zarr replacements"',
    )

    # ---------------------------------------------------------------
    # Validate every generated file before writing any modifications.
    # ---------------------------------------------------------------
    compile(helperSource, str(helperPath), "exec")
    generated = {editor.path: editor.render() for editor in editors.values()}
    generated[helperPath] = helperSource

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    backupDirectory = packagePath / f"_progress_backup_{stamp}"
    backupDirectory.mkdir(exist_ok=False)

    for editor in editors.values():
        (backupDirectory / editor.path.name).write_text(editor.source, encoding="utf-8")

    # Publish each complete generated source file atomically. Backups remain
    # available if installation is interrupted between files.
    for path, content in generated.items():
        temporaryPath = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w",
                encoding="utf-8",
                dir=packagePath,
                prefix=".progress-update-",
                suffix=".py",
                delete=False,
            ) as handle:
                temporaryPath = Path(handle.name)
                handle.write(content)
                handle.flush()
                os.fsync(handle.fileno())

            if path.exists():
                os.chmod(temporaryPath, path.stat().st_mode & 0o777)
            os.replace(temporaryPath, path)
            temporaryPath = None
        finally:
            if temporaryPath is not None:
                temporaryPath.unlink(missing_ok=True)

    print("Installed shared action progress.")
    print(f"Backups: {backupDirectory}")
    print("Restart the notebook kernel before running any actions.")


installSharedActionProgress()

IndentationError: unindent does not match any outer indentation level (pglMRI.py, line 2474)